## Compare POI visits for charging and non-charging days
Author: Callie Clark

* create 3 groups:  1) EV drivers on charging day, 2) EV drivers on non-charging day, 3) all others
* count number of POI visits and 


In [ ]:
!pip install pydeck -q -q
exec(open('../EV00_settings.py').read())
from shapely import wkt
from scipy.spatial import cKDTree

In [ ]:

with open(current_session_path+f'ev_driver_info/user_set_{current_city}_model.txt', "r") as file:
    set_string = file.read() # Read the string from the file
ev_driver_ls = list(ast.literal_eval(set_string)) # Convert the string back to a set

In [ ]:

from shapely import wkt
from scipy.spatial import cKDTree

fitness_wellness = ['Fitness and Recreational Sports Centers']
healthcare_pharmacies = ['Pharmacies and Drug Stores', 'Health and Personal Care Stores']
beauty = ['Personal Care Services', 'Hair Salons']
restaurants_cafes = ['Restaurants and Other Eating Places', 'Drinking Places (Alcoholic Beverages)', 'Food Services and Drinking Places', 'Special Food Services', 'Bakeries and Tortilla Manufacturing']
retail = ['Clothing Stores', 'Shoe Stores', 'Department Stores', 'Clothing and Clothing Accessories Stores', 'Sporting Goods, Hobby, and Musical Instrument Stores', 'Jewelry, Luggage, and Leather Goods Stores', 'Book Stores and News Dealers', 'Electronics and Appliance Stores', 'Furniture Stores', 'Home Furnishings Stores', 'Used Merchandise Stores', 'Office Supplies, Stationery, and Gift Stores', 'General Merchandise Stores, including Warehouse Clubs and Supercenters', #this could be in groceries? 
          'Other Miscellaneous Store Retailers', 'Florists']
grocery = ['Grocery Stores', 'Grocery and Food Stores', 'Supermarkets', 'Specialty Food Stores', 'Food and Beverage Stores', 'Beer, Wine, and Liquor Stores']
other_errands = ['Drycleaning and Laundry Services', 'Automotive Repair and Maintenance', 'Consumer Goods Rental', 'Personal and Household Goods Repair and Maintenance', 'Postal Service']
entertainment = ['Museums, Historical Sites, and Similar Institutions', 'Amusement Parks and Arcades', 'Other Amusement and Recreation Industries', 'Motion Picture and Video Industries', 'Performing Arts Companies', 'Arts, Entertainment, and Recreation']



################################################################################
# functions
################################################################################

def categorize_top_category(row):
    top_category = row['TOP_CATEGORY']
    sub_category = row['SUB_CATEGORY']

    if sub_category in healthcare_pharmacies:
        return 'Pharmacies'
    # elif sub_category in fitness_wellness:
    #     return 'Fitness'
    elif top_category in healthcare_pharmacies:
        return 'Pharmacies'
    elif top_category in beauty:
        return 'Beauty'
    elif top_category in restaurants_cafes:
        return 'Restaurants'
    elif top_category in retail:
        return 'Retail'
    elif top_category in grocery:
        return 'Grocery'
    elif top_category in entertainment:
        return 'Entertainment'
    elif top_category in other_errands:
        return 'Errands'

    else:
        return 'Other'
    



def format_poi_df(df_proj): 

    evcs_buffered=stop_gdf.copy()
    evcs_buffered["geometry"] = evcs_buffered.buffer(50)


    #output one: poi points that intersect with evcs
    try:
        poi=pd.read_csv(root+f'data/POI_data/POI_{current_city}_approved.csv.gz',index_col=0)
    except:
        poi=pd.read_csv(root+f'data/POI_data/POI_{current_city}_approved.csv',index_col=0)

    poi.drop(columns=['geometry'],inplace=True)#'Unnamed: 0',
    poi.rename(columns={'LATITUDE':'Latitude','LONGITUDE':'Longitude'},inplace=True)
    poi=poi[~poi.TOP_CATEGORY.isin(['Museums, Historical Sites, and Similar Institutions','Amusement Parks and Arcades'])] #remove parks and museums
    poi.drop(columns=['SAFEGRAPH_BRAND_IDS','REGION',
       'POSTAL_CODE', 'ISO_COUNTRY_CODE', 'PHONE_NUMBER',
       'BRANDS', 'STORE_ID','CATEGORY_TAGS', 'OPENED_ON', 'CLOSED_ON', 'TRACKING_CLOSED_SINCE'],
                     inplace=True)

    poi_proj=prepare_gdf(poi) #poi_points 
    joined_point_df = gpd.sjoin(evcs_buffered[['stop_index','cuebiq_id','stop_zoned_datetime','geometry']],poi_proj,  how="left", predicate="intersects")

    joined_point_df['Category'] = joined_point_df.apply(categorize_top_category, axis=1)
    joined_point_df=joined_point_df.reset_index()
    joined_point_df = joined_point_df.drop(columns=[c for c in joined_point_df.columns if c.startswith('index')], errors='ignore')
    joined_point_df=joined_point_df.to_crs('EPSG:4326')
    

    #joined_point_df.to_csv(root+f'data/POI_data/POI_{current_city}_points.csv')    #output standardized points poi here 
    
    evcs_buffered_unmatched=evcs_buffered[evcs_buffered.stop_index.isin(joined_point_df.stop_index)]
    
    #output two: poi points that intersect with evcs
    poly_poi=pd.read_csv(root+f'data/POI_data/{current_city}_entertainment_poi_polygon.csv',index_col=0)
    poly_poi['geometry'] = poly_poi['POLYGON_WKT'].apply(wkt.loads)
    poly_poi_geo = gpd.GeoDataFrame(poly_poi, geometry='geometry', crs=4326)
    poly_poi_gdf=poly_poi_geo.to_crs('EPSG:3857')
    
    joined_poly_df = gpd.sjoin(evcs_buffered_unmatched[['stop_index','cuebiq_id','stop_zoned_datetime','geometry']],poly_poi_gdf, how="inner", predicate="intersects")
    joined_poly_df=joined_poly_df.drop_duplicates(subset='PLACEKEY')
    joined_poly_df['Category'] = joined_poly_df.apply(categorize_top_category, axis=1)
    joined_poly_df=joined_poly_df.reset_index()
    joined_poly_df = joined_poly_df.drop(columns=[c for c in joined_poly_df.columns if c.startswith('index')],errors='ignore')
    joined_poly_df=joined_poly_df.to_crs('EPSG:4326')
    #does this create duplicate POIs for Poly and point 

    #joined_poly_df.to_csv(root+f'data/POI_data/POI_{current_city}_polygons.csv')    #output standardized points poi here 
   

    
    #create combined gdf 

    poi_comb=pd.concat([joined_point_df,joined_poly_df],axis=0)
    #poi_comb=poi_comb.to_crs('EPSG:4326')
    poi_comb=poi_comb[['PLACEKEY', 'LOCATION_NAME',  'TOP_CATEGORY', 'SUB_CATEGORY','Category',
       'NAICS_CODE', 'Latitude', 'Longitude', 'STREET_ADDRESS', 'CITY',
       # 'REGION', 'POSTAL_CODE', 'OPEN_HOURS',  'OPENED_ON','WKT_AREA_SQ_METERS',
       # 'CLOSED_ON', 'TRACKING_CLOSED_SINCE',
        'MSA', 'geometry','cuebiq_id']].copy()
    #poi_comb.to_csv(root+f'data/POI_data/{sub_str}{current_city}_POIs_by_EVCS_{max_distance_POI_to_EVCS+50}m.csv')

    

    evcs_buffered=evcs_buffered.to_crs('EPSG:4326')
    joined_df = gpd.sjoin(poi_comb[['PLACEKEY', 'LOCATION_NAME',  'TOP_CATEGORY', 'SUB_CATEGORY','Category',
       'NAICS_CODE', 'Latitude', 'Longitude','geometry']],evcs_buffered[['cuebiq_id','stop_zoned_datetime','geometry']], how="inner", predicate="intersects")
       
    return joined_df

In [ ]:
def find_pois_kdtree_then_sjoin(df_points, poi_points_gdf, poi_polygons_gdf,
                                max_distance=50):
    """
    Two-tier POI mapping:
    1) KDTree nearest-neighbor to point POIs within max_distance
    2) For points with no match, spatially join to polygon POIs
    """
    df_points.reset_index(inplace=True)
    gdf=prepare_gdf(df_points,proj=True)
    gdf = gdf.dropna(subset=['geometry'])
    
    poi_points = poi_points_gdf.to_crs('EPSG:3857').dropna(subset=['geometry'])
    poi_polygons = poi_polygons_gdf.to_crs('EPSG:3857').dropna(subset=['geometry'])

    # Make sure CRS match
    assert gdf.crs == poi_points.crs == poi_polygons.crs, "CRS do not match!"

    # =====================================================
    # ===============  TIER 1: KDTree match  ==============
    # =====================================================

    df_xy = np.column_stack((gdf.geometry.x, gdf.geometry.y))
    poi_xy = np.column_stack((poi_points.geometry.x, poi_points.geometry.y))

    tree = cKDTree(poi_xy)
    distances, idxs = tree.query(df_xy, k=1, distance_upper_bound=max_distance)


    # Fill KDTree results
    gdf["placekey"]  = np.nan
    gdf["category"]  = "no poi"
    gdf["name"]      = np.nan

    matched_mask = (distances != np.inf) & (idxs < len(poi_points))

    gdf.loc[matched_mask, "placekey"] = poi_points.iloc[idxs[matched_mask]]["PLACEKEY"].values
    gdf.loc[matched_mask, "category"] = poi_points.iloc[idxs[matched_mask]]["Category"].values
    gdf.loc[matched_mask, "name"]     = poi_points.iloc[idxs[matched_mask]]["LOCATION_NAME"].values

    # =====================================================
    # =======  TIER 2: Spatial join for unmatched ==========
    # =====================================================

    unmatched = gdf[gdf["placekey"].isna()].copy()
    matched   = gdf[~gdf["placekey"].isna()].copy()


    if len(unmatched) > 0:
        
        unmatched_clean = unmatched.drop(columns=['placekey', 'category', 'name'])
        
        sjoined = gpd.sjoin(
            unmatched_clean,
            poi_polygons[['PLACEKEY', 'Category', 'LOCATION_NAME', 'geometry']],
            how='left',
            predicate='within'
        )

        #sjoined = sjoined.groupby(level=0).first() #removing multiple parks within EVCS range
        sjoined = sjoined[~sjoined.index.duplicated(keep='first')]


        sjoined.rename(columns={
            'PLACEKEY': 'placekey',
            'Category': 'category',
            'LOCATION_NAME': 'name'
        }, inplace=True)

        sjoined = sjoined.drop(columns=[c for c in sjoined.columns if c.startswith('index_')], errors='ignore')

        # fill missing polygon matches
        sjoined['category'] = sjoined['category'].fillna('no poi')



        if(len(matched))==0:
            final=sjoined.copy()
        else:
            final = pd.concat([matched,
                       sjoined],
                      axis=0,ignore_index=True)
            

    else:
        final = gdf.copy()


    
    
    return final




In [ ]:
for current_city in ['Boston','SF','Seattle','Denver']:
    point_poi_df=pd.read_csv(root+f'data/POI_data/POI_{current_city}_points.csv')
    point_poi_df['geometry'] = point_poi_df['geometry'].apply(wkt.loads)
    point_poi_geo = gpd.GeoDataFrame(point_poi_df, geometry='geometry', crs="EPSG:4326") 

    poly_poi_df=pd.read_csv(root+f'data/POI_data/POI_{current_city}_polygons.csv')  
    poly_poi_df['geometry'] = poly_poi_df['geometry'].apply(wkt.loads)
    poly_poi_geo = gpd.GeoDataFrame(poly_poi_df, geometry='geometry', crs="EPSG:4326") 
    
    with open(current_session_path+f'ev_driver_info/user_set_{current_city}_model.txt', "r") as file:
        set_string = file.read() # Read the string from the file
    ev_driver_ls = list(ast.literal_eval(set_string)) # Convert the string back to a set
    
    df_stops=pd.DataFrame()
    # convert inputs to datetime
    
    start_dt = dt.strptime(start_date, '%Y%m%d')
    end_dt = dt.strptime(end_date, '%Y%m%d')
    current_date = start_dt

    while current_date <= end_dt:
        print(current_date)

        # print alert of what day is being processed
        date_str = str(current_date.strftime('%Y%m%d'))
        date_start = time.time()

        ev_charge_daily=pd.read_csv(current_session_path+f'evcs_session_stops/model/driver_EVCS_behavior_{current_city}_{date_str}.csv',index_col=0)
        ev_charge_daily_ls=list(ev_charge_daily.cuebiq_id.values)

        stop_df = pd.read_csv(f"{data_path}raw/stop/stop_data_{current_city}_{date_str}.csv.gz")
        tot_stops_dict=stop_df[['cuebiq_id','dwell_time_minutes']].groupby('cuebiq_id').count().to_dict()['dwell_time_minutes']
        stop_df_sub=stop_df[stop_df.transformation_type=='KEEP'] #remove uplevelled 

        stop_gdf=prepare_gdf(stop_df_sub)

        df_stops_iter = find_pois_kdtree_then_sjoin(
            stop_gdf,
            point_poi_geo,
            poly_poi_geo,
            max_distance=50
        )

        df_stops_iter['ev_driver'] = df_stops_iter['cuebiq_id'].isin(ev_driver_ls).astype(int)
        df_stops_iter['charge_day'] = df_stops_iter['cuebiq_id'].isin(ev_charge_daily_ls).astype(int)
        df_stops_iter['tot_stops'] = df_stops_iter['cuebiq_id'].map(tot_stops_dict)
        df_stops_iter['active_day']=(df_stops_iter['tot_stops']>0)*1

        df_stops_iter.to_csv(f'{current_city}_poi_stop_counts_{date_str}.csv')
        current_date = current_date + timedelta(days = 1)